# Sesion 11 - Tuning y experimentacion distribuida

**Producto:** mejor modelo seleccionado con validacion distribuida y guardado como artefacto reutilizable.

## Proposito de la sesion

Comparar modelos y combinaciones de hiperparametros con validacion distribuida en Spark ML, usando el mismo problema de prediccion de latencia construido en las sesiones anteriores.

## Resultado esperado

Al finalizar, el estudiante debe identificar el mejor candidato segun metricas de evaluacion, guardar el modelo ganador y conservar una tabla de resultados que documente la experimentacion realizada.

En las sesiones 9 y 10 entrenamos, guardamos y aplicamos un modelo de regresion para predecir la latencia promedio de la siguiente ventana temporal. En esta sesion comparamos configuraciones y algoritmos usando Spark ML para elegir el mejor modelo de forma reproducible.

## 1. Idea general

El flujo de experimentacion sera:

1. Reconstruir el dataset supervisado de latencia.
2. Separar un holdout cronologico para evaluar al final.
3. Ejecutar validacion distribuida en el bloque de entrenamiento.
4. Comparar metricas de los candidatos.
5. Guardar el mejor modelo en `../artifacts/models/ml_latency_pipeline_best`.

La validacion distribuida reparte combinaciones de parametros y folds como trabajos de Spark. En datasets pequenos se nota poco, pero el patron es el mismo para volumenes mayores.

## 2. Crear SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("s11-tuning-experimentacion-distribuida") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

spark

## 3. Parametros y rutas

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

BATCH_INPUT_PATHS = [
    "../artifacts/output/orden_eventos_observabilidad_p1",
    "../artifacts/output/orden_eventos_parquet",
    "/opt/artifacts/output/orden_eventos_observabilidad_p1",
    "/opt/artifacts/output/orden_eventos_parquet",
]

BASELINE_MODEL_PATHS = [
    "../artifacts/models/ml_latency_pipeline_lr",
    "/opt/artifacts/models/ml_latency_pipeline_lr",
]

BEST_MODEL_PATH = "../artifacts/models/ml_latency_pipeline_best"
EXPERIMENTS_PATH = "../artifacts/output/ml_latency_tuning_results"

RANDOM_SEED = 42
HOLDOUT_RATIO = 0.20

## 4. Helpers de datos

Mantenemos el mismo contrato de features de la sesion 9 para que los modelos sean comparables.

In [ ]:
def first_existing_path(paths):
    for path in paths:
        if Path(path).exists():
            return path
    return None


def load_observability_batch():
    source_path = first_existing_path(BATCH_INPUT_PATHS)

    if source_path:
        print(f"Fuente batch: {source_path}")
        return spark.read.parquet(source_path)

    print("No se encontraron Parquet reales. Se creara un dataset sintetico para la practica.")
    base = spark.range(0, 480)
    return base.select(
        (F.timestamp_seconds(F.lit(1746300000) + F.col("id") * 20)).alias("kafkaTimestamp"),
        (
            F.lit(120)
            + (F.col("id") % 12) * 8
            + F.when((F.col("id") % 41) == 0, 190).otherwise(0)
            + F.when((F.col("id") % 97) > 70, 45).otherwise(0)
            + (F.rand(seed=RANDOM_SEED) * 35)
        ).cast("long").alias("latencyMs"),
        F.lit("ordenes").alias("topic"),
        (F.col("id") % 3).cast("int").alias("partition"),
        F.col("id").cast("long").alias("offset"),
        F.lit("demo").alias("origen"),
        F.lit(True).alias("isValid")
    )


def build_latency_series(input_df):
    return input_df \
        .filter(F.col("kafkaTimestamp").isNotNull()) \
        .filter(F.col("latencyMs").isNotNull()) \
        .groupBy(F.window("kafkaTimestamp", "1 minute")) \
        .agg(
            F.count("*").alias("eventCount"),
            F.avg("latencyMs").alias("avgLatencyMs"),
            F.min("latencyMs").alias("minLatencyMs"),
            F.max("latencyMs").alias("maxLatencyMs"),
            F.stddev("latencyMs").alias("stdLatencyMs")
        ) \
        .withColumn("windowStart", F.col("window.start")) \
        .withColumn("windowEnd", F.col("window.end")) \
        .drop("window") \
        .na.fill({"stdLatencyMs": 0.0}) \
        .orderBy("windowStart")


def add_features_and_label(series_df):
    from pyspark.sql.window import Window

    w = Window.orderBy("windowStart")
    return series_df \
        .withColumn("label", F.lead("avgLatencyMs", 1).over(w)) \
        .withColumn("latencyRangeMs", F.col("maxLatencyMs") - F.col("minLatencyMs")) \
        .withColumn("hourOfDay", F.hour("windowStart")) \
        .withColumn("minuteOfHour", F.minute("windowStart")) \
        .select(
            "windowStart",
            "windowEnd",
            "eventCount",
            "avgLatencyMs",
            "minLatencyMs",
            "maxLatencyMs",
            "stdLatencyMs",
            "latencyRangeMs",
            "hourOfDay",
            "minuteOfHour",
            "label"
        ) \
        .na.drop()

## 5. Construir dataset supervisado

In [ ]:
df_raw = load_observability_batch()
serie_latency = build_latency_series(df_raw)
dataset_ml = add_features_and_label(serie_latency).cache()

row_count = dataset_ml.count()

if row_count < 30:
    print(f"Solo hay {row_count} filas para ML. Se usara una serie sintetica mas amplia para tuning.")
    synthetic_base = spark.range(0, 480)
    df_raw = synthetic_base.select(
        (F.timestamp_seconds(F.lit(1746300000) + F.col("id") * 20)).alias("kafkaTimestamp"),
        (
            F.lit(120)
            + (F.col("id") % 12) * 8
            + F.when((F.col("id") % 41) == 0, 190).otherwise(0)
            + F.when((F.col("id") % 97) > 70, 45).otherwise(0)
            + (F.rand(seed=RANDOM_SEED) * 35)
        ).cast("long").alias("latencyMs")
    )
    serie_latency = build_latency_series(df_raw)
    dataset_ml = add_features_and_label(serie_latency).cache()
    row_count = dataset_ml.count()

print("Filas para ML:", row_count)
dataset_ml.orderBy("windowStart").show(20, truncate=False)

## 6. Separar entrenamiento y holdout cronologico

La validacion interna selecciona parametros. El holdout cronologico se reserva para una evaluacion final mas honesta sobre ventanas posteriores.

In [ ]:
from pyspark.sql.window import Window

w = Window.orderBy("windowStart")
ordered = dataset_ml.withColumn("rowNumber", F.row_number().over(w))
total_rows = ordered.count()
train_cutoff = int(total_rows * (1 - HOLDOUT_RATIO))

train_full = ordered.filter(F.col("rowNumber") <= train_cutoff).drop("rowNumber").cache()
holdout = ordered.filter(F.col("rowNumber") > train_cutoff).drop("rowNumber").cache()

print("Filas train/validacion:", train_full.count())
print("Filas holdout final:", holdout.count())

train_full.select("windowStart", "avgLatencyMs", "label").show(5, truncate=False)
holdout.select("windowStart", "avgLatencyMs", "label").show(5, truncate=False)

## 7. Definir columnas y evaluadores

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

feature_cols = [
    "eventCount",
    "avgLatencyMs",
    "minLatencyMs",
    "maxLatencyMs",
    "stdLatencyMs",
    "latencyRangeMs",
    "hourOfDay",
    "minuteOfHour",
]

rmse_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")


def evaluate_predictions(predictions):
    return {
        "rmse": rmse_evaluator.evaluate(predictions),
        "mae": mae_evaluator.evaluate(predictions),
        "r2": r2_evaluator.evaluate(predictions),
    }

## 8. Experimento A: LinearRegression con CrossValidator

`CrossValidator` entrena varias combinaciones de hiperparametros y calcula la metrica promedio. Spark reparte esos entrenamientos como jobs distribuidos.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

assembler_lr = VectorAssembler(inputCols=feature_cols, outputCol="rawFeatures", handleInvalid="keep")
scaler_lr = StandardScaler(inputCol="rawFeatures", outputCol="features", withStd=True, withMean=True)
lr = LinearRegression(featuresCol="features", labelCol="label", predictionCol="prediction")

pipeline_lr = Pipeline(stages=[assembler_lr, scaler_lr, lr])

param_grid_lr = ParamGridBuilder() \
    .addGrid(lr.maxIter, [30, 80]) \
    .addGrid(lr.regParam, [0.0, 0.03, 0.10]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

cv_lr = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=param_grid_lr,
    evaluator=rmse_evaluator,
    numFolds=3,
    parallelism=2,
    seed=RANDOM_SEED
)

cv_model_lr = cv_lr.fit(train_full)
pred_lr = cv_model_lr.bestModel.transform(holdout).cache()
metrics_lr = evaluate_predictions(pred_lr)

metrics_lr

## 9. Revisar parametros del mejor modelo lineal

In [ ]:
best_lr_stage = cv_model_lr.bestModel.stages[-1]

print("Mejor LinearRegression")
print("maxIter:", best_lr_stage.getMaxIter())
print("regParam:", best_lr_stage.getRegParam())
print("elasticNetParam:", best_lr_stage.getElasticNetParam())
print("intercept:", best_lr_stage.intercept)

lr_cv_rows = []
for params, metric in zip(param_grid_lr, cv_model_lr.avgMetrics):
    lr_cv_rows.append((
        "linear_regression_cv",
        float(metric),
        int(params[lr.maxIter]),
        float(params[lr.regParam]),
        float(params[lr.elasticNetParam])
    ))

lr_cv_df = spark.createDataFrame(
    lr_cv_rows,
    ["experiment", "cvRmse", "maxIter", "regParam", "elasticNetParam"]
)

lr_cv_df.orderBy("cvRmse").show(truncate=False)

## 10. Experimento B: RandomForestRegressor con TrainValidationSplit

El bosque aleatorio captura no linealidades y no necesita escalado. Usamos `TrainValidationSplit` para reducir costo frente a otro cross validation completo.

In [ ]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import TrainValidationSplit

assembler_rf = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    seed=RANDOM_SEED
)

pipeline_rf = Pipeline(stages=[assembler_rf, rf])

param_grid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 60]) \
    .addGrid(rf.maxDepth, [3, 5]) \
    .addGrid(rf.minInstancesPerNode, [1, 3]) \
    .build()

tvs_rf = TrainValidationSplit(
    estimator=pipeline_rf,
    estimatorParamMaps=param_grid_rf,
    evaluator=rmse_evaluator,
    trainRatio=0.8,
    parallelism=2,
    seed=RANDOM_SEED
)

tvs_model_rf = tvs_rf.fit(train_full)
pred_rf = tvs_model_rf.bestModel.transform(holdout).cache()
metrics_rf = evaluate_predictions(pred_rf)

metrics_rf

## 11. Revisar parametros del mejor bosque

In [ ]:
best_rf_stage = tvs_model_rf.bestModel.stages[-1]
best_rf_num_trees = best_rf_stage.getNumTrees() if callable(best_rf_stage.getNumTrees) else best_rf_stage.getNumTrees

print("Mejor RandomForestRegressor")
print("numTrees:", best_rf_num_trees)
print("maxDepth:", best_rf_stage.getMaxDepth())
print("minInstancesPerNode:", best_rf_stage.getMinInstancesPerNode())

rf_validation_rows = []
for params, metric in zip(param_grid_rf, tvs_model_rf.validationMetrics):
    rf_validation_rows.append((
        "random_forest_tvs",
        float(metric),
        int(params[rf.numTrees]),
        int(params[rf.maxDepth]),
        int(params[rf.minInstancesPerNode])
    ))

rf_validation_df = spark.createDataFrame(
    rf_validation_rows,
    ["experiment", "validationRmse", "numTrees", "maxDepth", "minInstancesPerNode"]
)

rf_validation_df.orderBy("validationRmse").show(truncate=False)

## 12. Comparar candidatos en holdout

La seleccion final se hace con RMSE en el holdout cronologico. Tambien mostramos MAE y R2 para tener una lectura mas completa.

In [ ]:
candidate_rows = [
    (
        "linear_regression_cv",
        float(metrics_lr["rmse"]),
        float(metrics_lr["mae"]),
        float(metrics_lr["r2"]),
        f"maxIter={best_lr_stage.getMaxIter()}, regParam={best_lr_stage.getRegParam()}, elasticNetParam={best_lr_stage.getElasticNetParam()}"
    ),
    (
        "random_forest_tvs",
        float(metrics_rf["rmse"]),
        float(metrics_rf["mae"]),
        float(metrics_rf["r2"]),
        f"numTrees={best_rf_num_trees}, maxDepth={best_rf_stage.getMaxDepth()}, minInstancesPerNode={best_rf_stage.getMinInstancesPerNode()}"
    ),
]

comparison_df = spark.createDataFrame(
    candidate_rows,
    ["candidate", "holdoutRmse", "holdoutMae", "holdoutR2", "bestParams"]
)

comparison_df.orderBy("holdoutRmse").show(truncate=False)

best_candidate = comparison_df.orderBy("holdoutRmse").first()
print("Mejor candidato:", best_candidate["candidate"])

## 13. Comparar contra el modelo baseline de la sesion 9

Si existe el modelo original, lo evaluamos en el mismo holdout para saber si el tuning mejoro el resultado.

In [ ]:
from pyspark.ml import PipelineModel

baseline_path = first_existing_path(BASELINE_MODEL_PATHS)

if baseline_path:
    baseline_model = PipelineModel.load(baseline_path)
    baseline_predictions = baseline_model.transform(holdout)
    baseline_metrics = evaluate_predictions(baseline_predictions)
    print(f"Baseline cargado desde: {baseline_path}")
    print(baseline_metrics)
else:
    baseline_metrics = None
    print("No se encontro baseline de sesion 9. Se omite esta comparacion.")

## 14. Seleccionar y guardar el mejor modelo

Guardamos el `PipelineModel` completo. Asi conserva transformaciones de features y estimador final en un solo artefacto.

In [ ]:
if best_candidate["candidate"] == "linear_regression_cv":
    best_model = cv_model_lr.bestModel
    best_predictions = pred_lr
else:
    best_model = tvs_model_rf.bestModel
    best_predictions = pred_rf

best_model.write().overwrite().save(BEST_MODEL_PATH)

print(f"Mejor modelo guardado en: {BEST_MODEL_PATH}")
best_predictions.select(
    "windowStart",
    F.round("avgLatencyMs", 2).alias("currentAvgLatencyMs"),
    F.round("label", 2).alias("realNextAvgLatencyMs"),
    F.round("prediction", 2).alias("predictedNextAvgLatencyMs")
).orderBy("windowStart").show(20, truncate=False)

## 15. Guardar resultados de experimentacion

Esta salida sirve como evidencia del proceso de seleccion, no solo del modelo final.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

experiment_rows = candidate_rows

if baseline_metrics:
    experiment_rows.append((
        "baseline_session_09",
        float(baseline_metrics["rmse"]),
        float(baseline_metrics["mae"]),
        float(baseline_metrics["r2"]),
        "modelo guardado en sesion 9"
    ))

experiment_schema = StructType([
    StructField("candidate", StringType(), False),
    StructField("holdoutRmse", DoubleType(), False),
    StructField("holdoutMae", DoubleType(), False),
    StructField("holdoutR2", DoubleType(), False),
    StructField("bestParams", StringType(), False),
])

experiment_results_df = spark.createDataFrame(experiment_rows, experiment_schema) \
    .withColumn("selected", F.col("candidate") == F.lit(best_candidate["candidate"]))

experiment_results_df.write.mode("overwrite").parquet(EXPERIMENTS_PATH)

print(f"Resultados guardados en: {EXPERIMENTS_PATH}")
spark.read.parquet(EXPERIMENTS_PATH).orderBy("holdoutRmse").show(truncate=False)

## 16. Interpretacion de errores

Revisamos las ventanas con mayor error absoluto para entender donde el modelo falla.

In [ ]:
best_error_df = best_predictions \
    .withColumn("errorMs", F.col("label") - F.col("prediction")) \
    .withColumn("absErrorMs", F.abs(F.col("errorMs")))

best_error_df.select(
    "windowStart",
    "eventCount",
    F.round("avgLatencyMs", 2).alias("currentAvgLatencyMs"),
    F.round("label", 2).alias("realNextAvgLatencyMs"),
    F.round("prediction", 2).alias("predictedNextAvgLatencyMs"),
    F.round("errorMs", 2).alias("errorMs"),
    F.round("absErrorMs", 2).alias("absErrorMs")
).orderBy(F.desc("absErrorMs")).show(20, truncate=False)

## 17. Cargar el modelo ganador y probar reutilizacion

In [ ]:
loaded_best_model = PipelineModel.load(BEST_MODEL_PATH)
reloaded_predictions = loaded_best_model.transform(holdout)

print("Metricas del modelo recargado:")
print(evaluate_predictions(reloaded_predictions))

reloaded_predictions.select("windowStart", "label", "prediction").show(5, truncate=False)

## 18. Que observar en Spark UI

- Cada combinacion de parametros genera tareas de entrenamiento y evaluacion.
- `parallelism=2` permite evaluar combinaciones en paralelo cuando hay recursos disponibles.
- Los stages con `VectorAssembler`, escalado y entrenamiento aparecen como partes del pipeline.
- El holdout final se evalua despues de seleccionar parametros; no participa en la busqueda.

## 19. Cierre

Al terminar esta sesion debes tener:

- Comparacion de candidatos con RMSE, MAE y R2.
- Mejor modelo guardado en `../artifacts/models/ml_latency_pipeline_best`.
- Resultados de experimentos guardados en `../artifacts/output/ml_latency_tuning_results`.

Este modelo ganador puede reemplazar al modelo de la sesion 9 en la inferencia batch o streaming de la sesion 10.

## Ejercicios sugeridos

1. Agrega `GBTRegressor` como tercer candidato y compara su RMSE.
2. Cambia `numFolds` de 3 a 5 y observa el costo en Spark UI.
3. Agrega una feature `dayOfWeek` y revisa si mejora el holdout.
4. Reutiliza `ml_latency_pipeline_best` en la sesion 10 para inferencia streaming.